# Kaggle ASR v1 Workflow
Single-run workflow: setup, full v1 pipeline, and artifact export.

## Pre-run in Kaggle UI
- Settings -> Accelerator: GPU (T4)
- Settings -> Internet: ON
- Add Input datasets: code dataset + raw dataset.
- Run Save Version -> Save & Run All (Commit) to persist v1 outputs.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORKSPACE = Path('/kaggle/working/speech_recognation')
WORKSPACE.mkdir(parents=True, exist_ok=True)

def find_bundle_root(input_root: Path) -> Path:
    direct_candidates = sorted(
        path for path in input_root.rglob('kaggle_bundle')
        if path.is_dir() and (path / 'tools' / 'requirements_kaggle.txt').exists()
    )
    if direct_candidates:
        return direct_candidates[0]

    extracted_requirements = sorted(input_root.rglob('requirements_kaggle.txt'))
    for req_path in extracted_requirements:
        candidate_root = req_path.parent.parent
        if req_path.parent.name == 'tools' and (candidate_root / 'src').exists():
            return candidate_root

    zip_candidates = sorted(
        path for path in input_root.rglob('*.zip')
        if 'kaggle_bundle' in path.name.lower()
    )
    if zip_candidates:
        zip_path = zip_candidates[0]
        print('Extracting code zip from:', zip_path)
        with zipfile.ZipFile(zip_path, 'r') as archive:
            archive.extractall(WORKSPACE)
        extracted_root = WORKSPACE / 'kaggle_bundle'
        if (extracted_root / 'tools' / 'requirements_kaggle.txt').exists():
            return extracted_root

    raise FileNotFoundError(
        'Cannot locate kaggle_bundle under /kaggle/input. Attach the code dataset first.'
    )

BUNDLE_ROOT = find_bundle_root(INPUT_ROOT)
print('Detected Kaggle bundle root:', BUNDLE_ROOT)

required_dirs = ['src', 'tools', 'cli', 'configs']
for folder_name in required_dirs:
    source_dir = BUNDLE_ROOT / folder_name
    if not source_dir.exists():
        raise FileNotFoundError(f'Missing required folder inside bundle: {source_dir}')
    shutil.copytree(source_dir, WORKSPACE / folder_name, dirs_exist_ok=True)

optional_root_files = ['requirements.txt']
for file_name in optional_root_files:
    source_file = BUNDLE_ROOT / file_name
    if source_file.exists():
        shutil.copy2(source_file, WORKSPACE / file_name)

os.chdir(WORKSPACE)
PROJECT_ROOT = Path.cwd()
REQUIREMENTS_PATH = (PROJECT_ROOT / 'tools' / 'requirements_kaggle.txt').resolve()
PREPARE_ENV_PATH = (PROJECT_ROOT / 'tools' / 'prepare_environment.py').resolve()

assert REQUIREMENTS_PATH.exists(), f'Bootstrap failed: missing {REQUIREMENTS_PATH}'
assert PREPARE_ENV_PATH.exists(), f'Bootstrap failed: missing {PREPARE_ENV_PATH}'
assert (PROJECT_ROOT / 'cli' / 'build_aligned_chunks.py').exists(), 'Bootstrap failed: missing cli/build_aligned_chunks.py'
print('Workspace ready:', PROJECT_ROOT)
print('Requirements path:', REQUIREMENTS_PATH)

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REQUIREMENTS_PATH)], check=True)
subprocess.run([sys.executable, str(PREPARE_ENV_PATH)], check=True)

In [ ]:
from pathlib import Path

# If you know the exact path, set it here; leave None for auto-detection.
RAW_DIR = None
WORK_DIR_V1 = '/kaggle/working/asr_full_run_v1'

EPOCHS_V1 = 8
BATCH_SIZE_V1 = 8
LR_V1 = 2e-6

# Locked rerun defaults for stronger subgroup statistics + strict reproducibility.
HOLDOUT_RATIO = 0.30
FREEZE_HOLDOUT = True
QA_UNMATCHED_THRESHOLD = 0.42
MIN_ALIGNED_WORD_RATIO = 0.50
MAX_CHARS_PER_SECOND = 23.0

input_root = Path('/kaggle/input')
assert input_root.exists(), 'Missing /kaggle/input'

print('Datasets visible under /kaggle/input:')
for item in sorted(input_root.iterdir()):
    if item.is_dir():
        print(' -', item.name)

audio_suffixes = {'.mp3', '.wav', '.flac', '.m4a', '.ogg'}

def count_media(folder: Path) -> tuple[int, int]:
    audio_count = sum(1 for path in folder.rglob('*') if path.suffix.lower() in audio_suffixes)
    text_count = sum(1 for path in folder.rglob('*.txt'))
    return audio_count, text_count

def collect_candidates() -> list[tuple[Path, int, int]]:
    candidates: list[tuple[Path, int, int]] = []
    seen: set[Path] = set()

    for raw_dir in sorted(path for path in input_root.rglob('raw') if path.is_dir()):
        audio_count, text_count = count_media(raw_dir)
        if audio_count > 0 and text_count > 0 and raw_dir not in seen:
            candidates.append((raw_dir, audio_count, text_count))
            seen.add(raw_dir)

    if candidates:
        return candidates

    for folder in sorted(path for path in input_root.rglob('*') if path.is_dir()):
        audio_count, text_count = count_media(folder)
        if audio_count > 0 and text_count > 0 and folder not in seen:
            candidates.append((folder, audio_count, text_count))
            seen.add(folder)

    return candidates

candidates = collect_candidates()

if RAW_DIR is None:
    if not candidates:
        print('WARNING: raw audio dataset was not auto-detected under /kaggle/input.')
        manual_dataset = input(
            'Enter Kaggle dataset name or full path for the raw dataset, then press Enter: '
        ).strip()
        if not manual_dataset:
            raise FileNotFoundError(
                'No raw dataset provided. Attach the RTS/raw dataset or enter its path manually.'
            )

        manual_path = Path(manual_dataset)
        if not manual_path.exists():
            matching_roots = [path for path in input_root.rglob(manual_dataset) if path.is_dir()]
            if not matching_roots:
                raise FileNotFoundError(
                    f'Could not resolve dataset name or path: {manual_dataset}'
                )
            manual_path = matching_roots[0]

        RAW_DIR = str(manual_path.resolve())
        selected_path = Path(RAW_DIR)
        audio_count, text_count = count_media(selected_path)
    else:
        preferred = [item for item in candidates if 'speech-recognation-raw' in str(item[0]).lower()]
        selected_path, audio_count, text_count = preferred[0] if preferred else candidates[0]
        RAW_DIR = str(selected_path.resolve())
else:
    RAW_DIR = RAW_DIR.strip()
    selected_path = Path(RAW_DIR)
    audio_count, text_count = count_media(selected_path)

assert Path(RAW_DIR).exists(), f'RAW_DIR does not exist: {RAW_DIR}'
print('Selected RAW_DIR:', RAW_DIR)
print(f'Found files under RAW_DIR -> audio: {audio_count}, text: {text_count}')

if audio_count == 0 or text_count == 0:
    raise ValueError(
        f'RAW_DIR is not valid for training (audio={audio_count}, text={text_count}).'
    )

In [ ]:
import subprocess

cmd = [
    'python', 'tools/run_full_pipeline.py',
    '--raw_dir', RAW_DIR,
    '--work_dir', WORK_DIR_V1,
    '--epochs', str(EPOCHS_V1),
    '--batch_size', str(BATCH_SIZE_V1),
    '--learning_rate', str(LR_V1),
    '--holdout_ratio', str(HOLDOUT_RATIO),
    '--qa_unmatched_threshold', str(QA_UNMATCHED_THRESHOLD),
    '--min_aligned_word_ratio', str(MIN_ALIGNED_WORD_RATIO),
    '--max_chars_per_second', str(MAX_CHARS_PER_SECOND),
]

if FREEZE_HOLDOUT:
    cmd.append('--freeze_holdout')
else:
    cmd.append('--no-freeze_holdout')

print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
from pathlib import Path

summary_v1 = Path(WORK_DIR_V1) / 'metrics' / 'comparison_summary.json'
print(json.dumps(json.loads(summary_v1.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
!python tools/export_artifacts.py --source_dir /kaggle/working/asr_full_run_v1 --zip_prefix asr_full_run_v1

In [ ]:
import subprocess
from pathlib import Path

metrics_dir = Path(WORK_DIR_V1) / 'metrics'
baseline_preds = metrics_dir / 'baseline_holdout_predictions.jsonl'
finetuned_preds = metrics_dir / 'finetuned_holdout_predictions.jsonl'

check_cmd = [
    'python', 'tools/pre_eval_check.py',
    '--baseline', str(baseline_preds),
    '--finetuned', str(finetuned_preds),
]
print('RUN:', ' '.join(check_cmd))
subprocess.run(check_cmd, check=True)